# Loan Limit Increase Optimization (2023)

This notebook implements a hybrid analytics pipeline for optimizing loan limit increases:
- **Markov chain** risk-state transitions
- **Logistic demand model** for uptake forecasting
- **Expected-value / knapsack optimization** under capital constraints
- **Monte Carlo lifecycle simulation** for policy comparison

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path('..').resolve()
sys.path.append(str(ROOT / 'src'))

from data_loader import MACRO_SCENARIOS, load_data, prepare_features, PROFIT_PER_INCREASE, DISCOUNT_RATE_ANNUAL
from models import estimate_markov_matrix, fit_uptake_model, steady_state_distribution, expected_incremental_profit
from optimization import build_optimization_problem, compare_policies, macro_sensitivity

sns.set_theme(style='whitegrid')

## 1. Load and Explore Data

In [ ]:
df = prepare_features(load_data(ROOT / 'data' / 'loan_limit_increases.csv'))
print(f'Records: {len(df):,}')
df.head()

In [ ]:
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.histplot(df['increases_2023'], bins=6, ax=axes[0])
axes[0].set_title('Limit Increases in 2023')
sns.boxplot(data=df, x='risk_category', y='total_profit', ax=axes[1])
axes[1].set_title('Profit by Risk Tier')
sns.scatterplot(data=df.sample(2000, random_state=1), x='on_time_payment_pct', y='increases_2023', hue='risk_category', alpha=0.5, ax=axes[2])
axes[2].set_title('Repayment vs Increases')
plt.tight_layout()
plt.show()

## 2. Markov Chain Risk Transitions

In [ ]:
markov = estimate_markov_matrix(df)
steady = steady_state_distribution(markov)
print('Transition Matrix:')
display(markov)
print('\nSteady-State Distribution:')
display(steady)

In [ ]:
sns.heatmap(markov, annot=True, fmt='.2f', cmap='Blues')
plt.title('Risk-State Transition Probabilities')
plt.show()

## 3. Stochastic Demand Forecasting

In [ ]:
uptake = fit_uptake_model(df)
df['uptake_probability'] = uptake['uptake_probability']
print(f"Train accuracy: {uptake['train_accuracy']:.3f}")
print(f"Test accuracy: {uptake['test_accuracy']:.3f}")
df[['customer_id', 'on_time_payment_pct', 'took_increases', 'uptake_probability']].head()

## 4. Expected Profit and Optimization

In [ ]:
df['expected_profit'] = df.apply(lambda r: expected_incremental_profit(r, r['uptake_probability']), axis=1)
capital_budget = df.loc[df['eligible'], 'exposure'].sum() * 0.35
opt = build_optimization_problem(df, df['uptake_probability'], capital_budget=capital_budget)

print(f"Capital budget: ${capital_budget:,.0f}")
print(f"Customers selected: {opt['n_selected']:,}")
print(f"Expected profit: ${opt['expected_profit']:,.0f}")
print(f"Total exposure: ${opt['total_exposure']:,.0f}")

## 5. Monte Carlo Policy Simulation

In [ ]:
policy_results = compare_policies(df, df['uptake_probability'])
policy_results

In [ ]:
sns.barplot(data=policy_results, x='policy', y='mean_profit', hue='policy', legend=False, palette='viridis')
plt.title('Mean Simulated Profit by Policy')
plt.ylabel('Profit ($)')
plt.show()

## 6. Macroeconomic Sensitivity

In [ ]:
macro_results = macro_sensitivity(df, df['uptake_probability'], MACRO_SCENARIOS, capital_budget)
macro_results

## 7. Key Takeaways

1. **Optimized selective offering** dominates aggressive mass offering in simulated profit.
2. **Risk migration** concentrates ~56% of long-run customers in subprime without intervention.
3. **Macro stress** reduces uptake and expected profit by ~7% in adverse conditions.
4. Operationalize via monthly scoring, eligibility rules, and capital-aware offer lists.